In [ ]:
!pip install -q rfdetr

In [ ]:
pip install onnxruntime-gpu

In [1]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
import os 
from accuracy_indian import compute_metrics
from generate_isolated_masks import generate_isolated_mask
from rfdetr import RFDETRBase
from PIL import Image

2025-05-28 15:31:42.574347: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 15:31:42.580496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748426502.587816   15657 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748426502.590041   15657 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748426502.595654   15657 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
# Load SAM model
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/SAM_MODEL/Final_Models/FineTune_model_epoch_30_27_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_DETR_SAM_PreTrain.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

images_dirs = sorted(os.listdir(data_folder))
masks_dirs = sorted(os.listdir(mask_folder))


checkpoint_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/indian_weightss/detr_indian.pth"  

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_wrapper = RFDETRBase()

underlying_model = model_wrapper.model
underlying_model.reinitialize_detection_head(num_classes=2)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pytorch_model = underlying_model.model
pytorch_model.load_state_dict(checkpoint['model'])

pytorch_model.to(device)
pytorch_model.eval()

for i in range(len(os.listdir(data_folder))):
    img_name = images_dirs[i]
    mask_name = masks_dirs[i]

    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    detections = model_wrapper.predict(image, threshold=0.5)
    boxes = detections.xyxy
    
    mask_path = os.path.join(mask_folder, mask_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

Loading pretrain weights


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

metrics = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_DETR_SAM_PreTrain.csv")

output = metrics.mean(axis=0)
output

pixel_iou                  0.747289
pixel_dice                 0.833635
pixel_accuracy             0.999033
pixel_precision            0.791122
pixel_recall               0.928418
region_iou                 0.747289
region_dice                0.833635
region_precision           0.791122
region_recall              0.928418
region_success_accuracy    0.939195
dtype: float64